## SLASCONE licensing

Container creation:
- Set the SLASCONE license ID in an environment variable
- Expose the Jupyter Lab port 8888
- Inject the home directory as volume

```
docker run --name=d2h-poc --env=SLAS_LICENSE_ID=683f0de2-ea30-46a6-9596-6b03541bec2d --volume=C:\Projects\jovyan:/home/jovyan -p 8888:8888 -d jupyter/scipy-notebook:latest
```

#### Get the hostname

If not overwritten on container creation the hostname is the trunkated container ID

In [1]:
import socket
print(socket.gethostname())

292d780b00a2


#### Get system/container infos

In [2]:
import platform
uname = platform.uname()
print(uname)

client_name = uname.system
print(client_name)
client_description = f"{uname.release} {uname.machine}"
print(client_description)

uname_result(system='Linux', node='292d780b00a2', release='6.6.87.2-microsoft-standard-WSL2', version='#1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025', machine='x86_64')
Linux
6.6.87.2-microsoft-standard-WSL2 x86_64


#### Read the license ID from the environment variable

In [17]:
import os
# licenseID = os.environ['SLAS_LICENSE_ID']
licenseID = '27180460-29df-4a5a-a0a1-78c85ab6cee0'
print(licenseID)

27180460-29df-4a5a-a0a1-78c85ab6cee0


#### Extract the container ID from `/proc/self/mountinfo`

In [4]:
with open('/proc/self/mountinfo') as file:
    line = file.readline().strip()
    while line:
        if '/docker/containers/' in line:
            containerID = line.split('/docker/containers/')[-1]
            containerID = containerID.split('/')[0]
            print(f"Container ID: {containerID}")
        line = file.readline().strip()

Container ID: 292d780b00a221d727d9565d95767e166bc76ae37218674b9b874a3f96479c07
Container ID: 292d780b00a221d727d9565d95767e166bc76ae37218674b9b874a3f96479c07
Container ID: 292d780b00a221d727d9565d95767e166bc76ae37218674b9b874a3f96479c07


#### Set SLASCONE variables

- Base URL of the Rest API backend
- Independent Software Vendor (ISV) ID
- ID of the licensed product
- Provisioning key to authenticate the client to the backend

In [18]:
# Set Slascone variables
baseUrl = "https://api.slascone.com"
isvId = "2af5fe02-6207-4214-946e-b00ac5309f53"
productId = "b18657cc-1f7c-43fa-e3a4-08da6fa41ad3"
provisioningKey = "NfEpJ2DFfgczdYqOjvmlgP2O/4VlqmRHXNE9xDXbqZcOwXTbH3TFeBAKKbEzga7D7ashHxFtZOR142LYgKWdNocibDgN75/P58YNvUZafLdaie7eGwI/2gX/XuDPtqDW"

#### Function to post requests to the Rest API

The function also sets the authentication header.

In [6]:
import requests
import json

def post_request(endpoint, data):
    # API endpoint URL
    url = f"{baseUrl}/api/v2/isv/{isvId}/{endpoint}"

    # Authorization header
    headers = {
        "ProvisioningKey": provisioningKey,
        "Content-Type": "application/json"
    }
    
    # Make the POST request
    response = requests.post(url, json=data, headers=headers)

    # Print the response
    print(f"Response status code: {response.status_code}")
    print(f"Response content: {json.dumps(json.loads(response.content.decode('utf-8')), indent=4 )}")

#### Create a license heartbeat

This should always be the first step.

If the license is activated for the client the client will receive all license info.
- Validity of the license
- Provisioning mode
- Features
- Limitation
- Variables
- etc.

In [19]:
# JSON data to send
data = {
  "product_id": productId,
  "client_id": containerID,
  "software_version": "2026.6",
  "operating_system": client_name
}

post_request("provisioning/heartbeats", data)

Response status code: 409
Response content: {
    "id": 2006,
    "message": "Unknown client.",
    "help": ""
}


#### Activate the license

In case the license heartbeat runs into an error "Unknown client ID" the license first has to be activated.

With the activation you can submit additional information about the client like the operating system in use or version numbers.

In [20]:
# JSON data to send
data = {
  "product_id": productId,
  "license_key": licenseID,
  "client_id": containerID,
  "client_name": client_name,
  "client_description": client_description,
  "software_version": "2024.1"
}

post_request("provisioning/activations", data)

Response status code: 200
Response content: {
    "license_key": "27180460-29df-4a5a-a0a1-78c85ab6cee0",
    "license_name": "New License",
    "legacy_license_key": null,
    "token_key": "4d6238cc-cb08-43c5-a5db-c6277212158f",
    "license_type": {
        "id": "0e30e2b1-c3b6-467f-b819-c122bf54928f",
        "name": "Postman Demo Test",
        "created_date_utc": "2022-09-20T04:42:48Z",
        "modified_date_utc": "2022-09-20T04:42:48Z",
        "last_modified_by": "operations@slascone.com"
    },
    "is_license_active": true,
    "is_license_expired": false,
    "is_license_valid": true,
    "start_date_utc": "2022-09-20T04:53:24Z",
    "expiration_date_utc": "2039-12-31T00:00:00Z",
    "date_validity": 0,
    "customer": {
        "customer_id": "103aba57-4c33-494c-bdd5-fd345bc683f5",
        "company_name": "Ring Retail",
        "customer_number": "87012"
    },
    "product_id": "b18657cc-1f7c-43fa-e3a4-08da6fa41ad3",
    "product_name": "CAD",
    "template_id": "938e1af5-c

#### Generate a session ID

If the provisioning mode of the license is _floating_ the next step is to open a session.

In [21]:
import uuid

# Generate a random UUID for session Id
sessionID = str(uuid.uuid4())

In [22]:
# Print the session Id
print(sessionID)

fcb63090-541c-4fd7-9ddd-7337e6c7afde


#### Open a session

In [28]:
# JSON data to send
data = {
  "license_id": licenseID,
  "client_id": containerID,
  "session_id": sessionID
}

post_request("provisioning/session/open", data)

Response status code: 200
Response content: {
    "session_id": "fcb63090-541c-4fd7-9ddd-7337e6c7afde",
    "limitation_id": null,
    "is_session_valid": true,
    "session_valid_until": "2026-11-25T09:16:06Z",
    "session_created_date": "2026-11-25T09:11:29Z",
    "session_modified_date": "2026-11-25T09:15:06Z",
    "max_open_session_count": 1,
    "max_active_client_count": 1
}


#### Close a session

In [29]:
data = {
  "license_id": licenseID,
  "client_id": containerID,
  "session_id": sessionID
}

post_request("provisioning/session/close", data)

Response status code: 200
Response content: "Success"
